# 1. Setup & Load Data

In [17]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [18]:
df = pd.read_csv(
    '../data/processed/online_retail_cleaned.csv'
)

C:\Users\user\AppData\Local\Temp\ipykernel_420\2852433281.py:1: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [19]:
df['invoice_no_normalized'] = (
    df['InvoiceNo']
      .astype(str)
      .str.strip()
)

In [20]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [21]:
print("Rows :", f"{len(df):,}")
print("Columns :", df.shape[1])

print(
    "Unique normalized invoices :",
    f"{df['invoice_no_normalized'].nunique():,}"
)

print(
    "Unique customers :",
    f"{df['CustomerID'].nunique():,}"
)

display(df.head())

Rows : 524,878
Columns : 11
Unique normalized invoices : 19,960
Unique customers : 4,338


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Revenue,invoice_no_normalized
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom,False,15.30,536365
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,20.34,536365
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom,False,22.00,536365
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,20.34,536365
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,20.34,536365


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 524878 entries, 0 to 524877
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   InvoiceNo              524878 non-null  object        
 1   StockCode              524878 non-null  str           
 2   Description            524878 non-null  str           
 3   Quantity               524878 non-null  int64         
 4   InvoiceDate            524878 non-null  datetime64[us]
 5   UnitPrice              524878 non-null  float64       
 6   CustomerID             392692 non-null  float64       
 7   Country                524878 non-null  str           
 8   is_cancelled           524878 non-null  bool          
 9   Revenue                524878 non-null  float64       
 10  invoice_no_normalized  524878 non-null  str           
dtypes: bool(1), datetime64[us](1), float64(3), int64(1), object(1), str(4)
memory usage: 40.5+ MB


# 2. Define Analysis Date

In [23]:
latest_transaction_date = df['InvoiceDate'].max()

print(
    "Latest transaction date :",
    latest_transaction_date
)

Latest transaction date : 2011-12-09 12:50:00


In [24]:
analysis_date = latest_transaction_date + pd.Timedelta(days=1)

print(
    "Analysis date :",
    analysis_date
)

Analysis date : 2011-12-10 12:50:00


# 3. Calculate Recency

In [26]:
customer_last_purchase = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID', as_index=False)
    .agg(
        last_purchase_date=('InvoiceDate', 'max')
    )
)

customer_last_purchase.head(10)

,CustomerID,last_purchase_date
0,"12,346.00",2011-01-18 10:01:00
1,"12,347.00",2011-12-07 15:52:00
2,"12,348.00",2011-09-25 13:13:00
3,"12,349.00",2011-11-21 09:51:00
4,"12,350.00",2011-02-02 16:01:00
5,"12,352.00",2011-11-03 14:37:00
6,"12,353.00",2011-05-19 17:47:00
7,"12,354.00",2011-04-21 13:11:00
8,"12,355.00",2011-05-09 13:49:00
9,"12,356.00",2011-11-17 08:40:00


In [27]:
customer_last_purchase['recency'] = (
    analysis_date - customer_last_purchase['last_purchase_date']
).dt.days

In [28]:
customer_last_purchase.head(10)

,CustomerID,last_purchase_date,recency
0,"12,346.00",2011-01-18 10:01:00,326
1,"12,347.00",2011-12-07 15:52:00,2
2,"12,348.00",2011-09-25 13:13:00,75
3,"12,349.00",2011-11-21 09:51:00,19
4,"12,350.00",2011-02-02 16:01:00,310
5,"12,352.00",2011-11-03 14:37:00,36
6,"12,353.00",2011-05-19 17:47:00,204
7,"12,354.00",2011-04-21 13:11:00,232
8,"12,355.00",2011-05-09 13:49:00,214
9,"12,356.00",2011-11-17 08:40:00,23


In [29]:
customer_last_purchase['recency'].describe()

count   4,338.00
mean       92.54
std       100.01
min         1.00
25%        18.00
50%        51.00
75%       142.00
max       374.00
Name: recency, dtype: float64

# 4. Calculate Frequency

In [34]:
customer_frequency = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID', as_index=False)
    .agg(
        frequency=('invoice_no_normalized', 'nunique')
    )
)

customer_frequency.head(10)

,CustomerID,frequency
0,"12,346.00",1
1,"12,347.00",7
2,"12,348.00",4
3,"12,349.00",1
4,"12,350.00",1
5,"12,352.00",8
6,"12,353.00",1
7,"12,354.00",1
8,"12,355.00",1
9,"12,356.00",3


In [35]:
customer_frequency['frequency'].describe()

count   4,338.00
mean        4.27
std         7.70
min         1.00
25%         1.00
50%         2.00
75%         5.00
max       209.00
Name: frequency, dtype: float64

# 5. Calculate Monetary

In [42]:
customer_monetary = (
    df[df['CustomerID'].notna()]
    .groupby('CustomerID', as_index=False)
    .agg(
        monetary=('Revenue', 'sum')
    )
)

customer_monetary.head(10)

,CustomerID,monetary
0,"12,346.00","77,183.60"
1,"12,347.00","4,310.00"
2,"12,348.00","1,797.24"
3,"12,349.00","1,757.55"
4,"12,350.00",334.40
5,"12,352.00","2,506.04"
6,"12,353.00",89.00
7,"12,354.00","1,079.40"
8,"12,355.00",459.40
9,"12,356.00","2,811.43"


In [43]:
customer_monetary['monetary'].describe()

count     4,338.00
mean      2,048.69
std       8,985.23
min           3.75
25%         306.48
50%         668.57
75%       1,660.60
max     280,206.02
Name: monetary, dtype: float64

# 6. RFM Scoring

In [49]:
rfm = (
    customer_last_purchase[
        ['CustomerID', 'recency']
    ]
    .merge(
        customer_frequency[
            ['CustomerID', 'frequency']
        ],
        on='CustomerID',
        how='inner'
    )
    .merge(
        customer_monetary[
            ['CustomerID', 'monetary']
        ],
        on='CustomerID',
        how='inner'
    )
)

rfm.head(10)

,CustomerID,recency,frequency,monetary
0,"12,346.00",326,1,"77,183.60"
1,"12,347.00",2,7,"4,310.00"
2,"12,348.00",75,4,"1,797.24"
3,"12,349.00",19,1,"1,757.55"
4,"12,350.00",310,1,334.40
5,"12,352.00",36,8,"2,506.04"
6,"12,353.00",204,1,89.00
7,"12,354.00",232,1,"1,079.40"
8,"12,355.00",214,1,459.40
9,"12,356.00",23,3,"2,811.43"


In [50]:
print("RFM rows :", f"{len(rfm):,}")
print("RFM columns :", rfm.shape[1])

RFM rows : 4,338
RFM columns : 4


In [51]:
print(
    "Expected customers :",
    f"{df['CustomerID'].nunique():,}"
)

print(
    "RFM customers      :",
    f"{rfm['CustomerID'].nunique():,}"
)

print(
    "Customer count match :",
    rfm['CustomerID'].nunique()
    == df['CustomerID'].nunique()
)

Expected customers : 4,338
RFM customers      : 4,338
Customer count match : True


In [52]:
print(
    "Duplicate customers :",
    rfm['CustomerID'].duplicated().sum()
)

Duplicate customers : 0


In [53]:
print(
    rfm[
        ['recency', 'frequency', 'monetary']
    ].isna().sum()
)

recency      0
frequency    0
monetary     0
dtype: int64


## 6.1 RFM Score - Recency

In [54]:
rfm['R_score'] = pd.qcut(
    rfm['recency'],
    q=5,
    labels=[5,4,3,2,1],
    duplicates='drop'
).astype(int)

In [55]:
rfm['R_score'].value_counts().sort_index()

R_score
1    865
2    843
3    858
4    904
5    868
Name: count, dtype: int64

In [56]:
print(
    rfm.groupby('R_score')['recency']
    .agg(['min', 'max', 'mean', 'count'])
    .sort_index(ascending=False)
)

         min  max   mean  count
R_score                        
5          1   13   6.17    868
4         14   33  23.06    904
3         34   72  52.32    858
2         73  179 116.25    843
1        180  374 268.60    865


## 6.2 RFM Score — Frequency

In [60]:
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [61]:
rfm['F_score'].value_counts().sort_index()

F_score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

In [64]:
f_score_means = (
    rfm.groupby('F_score')['frequency']
    .mean()
)

print(f_score_means)

F_score
1    1.00
2    1.28
3    2.32
4    4.02
5   12.74
Name: frequency, dtype: float64


## 6.3 RFM Score - Monetary

In [65]:
rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    q=5,
    labels=[1,2,3,4,5],
    duplicates='drop'
).astype(int)

In [67]:
rfm['M_score'].value_counts().sort_index()

M_score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

In [68]:
m_score_means = (
    rfm.groupby('M_score')['monetary']
    .mean()
)

print(m_score_means)

M_score
1     152.58
2     357.56
3     684.34
4   1,399.61
5   7,646.66
Name: monetary, dtype: float64


## 6.4 Calculate RFM Score

In [71]:
rfm['RFM_score'] = (
    rfm['R_score']
    + rfm['F_score']
    + rfm['M_score']
)

In [72]:
rfm['RFM_score'].value_counts().sort_index()

RFM_score
3     183
4     361
5     337
6     426
7     377
8     375
9     336
10    342
11    347
12    321
13    286
14    300
15    347
Name: count, dtype: int64

In [74]:
rfm[
    [
        'CustomerID',
        'recency',
        'frequency',
        'monetary',
        'R_score',
        'F_score',
        'M_score',
        'RFM_score'
    ]
].sort_values(
    'RFM_score',
    ascending=False
).head(20)

,CustomerID,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
4309,"18,245.00",7,7,"2,567.06",5,5,5,15
4307,"18,241.00",10,17,"2,073.09",5,5,5,15
4298,"18,230.00",9,7,"2,810.20",5,5,5,15
4297,"18,229.00",12,20,"7,276.90",5,5,5,15
4293,"18,225.00",3,12,"5,504.96",5,5,5,15
4291,"18,223.00",5,14,"6,484.54",5,5,5,15
4287,"18,219.00",3,10,"2,069.77",5,5,5,15
4279,"18,210.00",2,6,"2,621.38",5,5,5,15
4272,"18,198.00",4,17,"5,425.56",5,5,5,15
4231,"18,144.00",8,12,"2,888.75",5,5,5,15
